In [1]:
%env PYTORCH_ENABLE_MPS_FALLBACK=1
%cd ..

env: PYTORCH_ENABLE_MPS_FALLBACK=1
/Users/n-zagainov/kolobok/ml


In [14]:
import json
from pathlib import Path
import sys
import shutil

import numpy as np
import cv2
import pandas as pd

from matplotlib import pyplot as plt

import torch
from torch import nn
from torch.nn import functional as F
from torchvision.io import read_image, write_png
from torchvision.transforms import functional as VF, InterpolationMode

from tqdm import tqdm

from tire_vision.thread.segmentation.segmentator import SegmentationInferencer
from tire_vision.config import SegmentatorConfig

src_root = Path("data/raw")
dst_root = Path("data/thread_seg")

In [3]:
cfg = SegmentatorConfig(
    device="mps",
    target="wheel-tire-thread",
)

seg = SegmentationInferencer(cfg)


Loading config /Users/n-zagainov/kolobok/ml/external/SAN/configs/Base-coco-stuff-164K-171.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
/Users/n-zagainov/kolobok/ml/external/SAN/san/model/side_adapter/timm_wrapper.py:24: UserWarning: Unused kwargs are provided:{'dynamic_img_pad': False}.
  warnings.warn(f"Unused kwargs are provided:{kwargs}.")
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
img = read_image("/Users/n-zagainov/kolobok/ml/data/raw/1,2мм/20220907_110649.jpg").to("mps")
result = seg(img).cpu()

/Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/torch/nn/functional.py:4704: UserWarning: The operator 'aten::_upsample_bicubic2d_aa.out' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:14.)
  return torch._C._nn._upsample_bicubic2d_aa(


In [28]:
input_images = list(src_root.glob("**/*.jpg")) + list(src_root.glob("**/*.JPG"))

In [40]:
for i, img_path in enumerate(tqdm(input_images)):
    img = read_image(img_path).to("mps")
    mask = seg(img).cpu() * 255
    img = img.cpu()

    assert img.shape[-2:] == mask.shape[-2:]

    img_save_path = dst_root / "images" / f"{i}.png"
    seg_save_path = dst_root / "masks" / f"{i}.png"

    img_save_path.parent.mkdir(parents=True, exist_ok=True)
    seg_save_path.parent.mkdir(parents=True, exist_ok=True)

    write_png(img, img_save_path)
    # write_png(mask[None], seg_save_path)
    cv2.imwrite(str(seg_save_path), mask.cpu().numpy())

100%|██████████| 1448/1448 [30:10<00:00,  1.25s/it]


In [39]:
mask.max()

tensor(1, dtype=torch.uint8)

In [4]:
def rembg(image: torch.Tensor) -> torch.Tensor:
    mask = seg(image.to("cuda")).cpu()
    return image * mask + 255 * (1 - mask)


def crop(image: torch.Tensor, padding_frac: float = 0.01) -> torch.Tensor:
    *_, h, w = image.shape
    padding = int(h * padding_frac), int(w * padding_frac)
    mask = seg(image.to("cuda")).cpu()
    image = image * mask + 255 * (1 - mask)
    i, j = torch.where(mask == 1)

    min_i, max_i = torch.min(i), torch.max(i)
    min_j, max_j = torch.min(j), torch.max(j)
    min_i = max(0, min_i - padding[0])
    max_i = min(h, max_i + padding[0])
    min_j = max(0, min_j - padding[1])
    max_j = min(w, max_j + padding[1])

    return image[..., min_i:max_i, min_j:max_j]

In [5]:
if dst_root.exists():
    shutil.rmtree(dst_root)
dst_root.mkdir(parents=True, exist_ok=True)

data = []

for dir in src_root.iterdir():
    label = float(dir.name[:-2].replace(",", "."))
    paths = list(dir.iterdir())
    loop = tqdm(paths, desc=f"Segmenting {label}mm tires")
    for img_path in loop:
        try:
            image = read_image(img_path)
            image = crop(image)

            save_name = f"{len(data)}.png"
            save_path = dst_root / save_name

            data.append([save_name, label])
            write_png(image, save_path)
        except Exception as e:
            print(f"Error: {e}, image: {img_path}")


df = pd.DataFrame(data=data, columns=["path", "label"])
df.to_csv(dst_root / "thread_depths.csv", index=False)

Segmenting 1.8mm tires: 100%|██████████| 2/2 [00:00<00:00,  2.14it/s]
Segmenting 9.4mm tires: 0it [00:00, ?it/s]
Segmenting 2.0mm tires: 100%|██████████| 15/15 [00:07<00:00,  2.04it/s]
Segmenting 9.6mm tires: 0it [00:00, ?it/s]
Segmenting 8.3mm tires:  36%|███▌      | 5/14 [00:01<00:03,  2.76it/s]

Error: min(): Expected reduction dim to be specified for input.numel() == 0. Specify the reduction dim with the 'dim' argument., image: data/dataset/8,3мм/20241204_234902.jpg


Segmenting 3.6mm tires: 100%|██████████| 25/25 [00:12<00:00,  1.93it/s]
Segmenting 9.7mm tires: 0it [00:00, ?it/s]
Segmenting 8.1mm tires: 100%|██████████| 4/4 [00:01<00:00,  2.13it/s]


In [6]:
# clahe = cv2.createCLAHE(clipLimit=5)
# img_path = dst_root / df.sample().iloc[0, 0]

# gray = VF.rgb_to_grayscale(read_image(img_path))[0].numpy()
# grad_x = cv2.Sobel(gray, cv2.CV_16S, 1, 0, ksize=3, scale=1, delta=0, borderType=cv2.BORDER_DEFAULT)
# grad_y = cv2.Sobel(gray, cv2.CV_16S, 0, 1, ksize=3, scale=1, delta=0, borderType=cv2.BORDER_DEFAULT)

# abs_grad_x = cv2.convertScaleAbs(grad_x)
# abs_grad_y = cv2.convertScaleAbs(grad_y)

# grad = cv2.addWeighted(abs_grad_x, 0.5, abs_grad_y, 0.5, 0)
# plt.imshow(grad, cmap="gray")